In [1]:
import os
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. SETUP PATH FOLDER
PROCESSED_CSV = os.path.abspath("data/processed/cases.csv")
EVAL_FOLDER = os.path.abspath("data/eval")
os.makedirs(EVAL_FOLDER, exist_ok=True)

# Load dataset terstruktur dari Tahap 2
df = pd.read_csv(PROCESSED_CSV)

# =====================================================================
# 2. SPLITTING DATA (Rasio 80:20)
# =====================================================================
# Data train akan disimpan sebagai Case Base tetap, data test sebagai simulasi kasus baru
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42, stratify=df['pasal'])

print(f"Jumlah Kasus Lama di Case Base (Train): {len(df_train)}")
print(f"Jumlah Kueri Kasus Baru untuk Uji Coba (Test): {len(df_test)}")

# =====================================================================
# 3. VEKTORISASI TEXT (TF-IDF APPROACH)
# =====================================================================
# Kita gunakan kolom 'ringkasan_fakta' sebagai fitur utama pencarian kemiripan
vectorizer = TfidfVectorizer(max_features=5000, stop_words=None) 

# Fit & transform pada data Case Base (Train)
train_vectors = vectorizer.fit_transform(df_train['ringkasan_fakta'])

# =====================================================================
# 4. IMPLEMENTASI FUNGSI RETRIEVAL
# =====================================================================
def retrieve(query: str, k: int = 5):
    """
    Fungsi untuk mencari top-k kasus paling mirip berdasarkan query teks baru
    """
    # 1) Pre-process query (cleaning sederhana)
    query_cleaned = query.lower()
    
    # 2) Hitung vektor TF-IDF untuk query baru
    query_vector = vectorizer.transform([query_cleaned])
    
    # 3) Hitung Cosine Similarity antara kueri dengan seluruh data di Case Base
    similarities = cosine_similarity(query_vector, train_vectors).flatten()
    
    # 4) Ambil indeks top-k dengan skor kemiripan tertinggi
    top_k_indices = np.argsort(similarities)[::-1][:k]
    
    # Ambil data kasus berdasarkan indeks teratas yang terpilih
    top_k_cases = df_train.iloc[top_k_indices]
    top_k_scores = similarities[top_k_indices]
    
    results = []
    for idx, (_, row) in enumerate(top_k_cases.iterrows()):
        results.append({
            "case_id": int(row['case_id']),
            "no_perkara": row['no_perkara'],
            "pasal_asli": row['pasal'],
            "similarity_score": float(top_k_scores[idx])
        })
        
    return results

# =====================================================================
# 5. PEMBUATAN FILE EVALUASI (queries.json)
# =====================================================================
# Mengambil 5 kasus dari data test untuk dijadikan bahan evaluasi formal
queries_eval = []

for _, row in df_test.head(5).iterrows():
    # Cari ground truth: Kasus di Train yang memiliki pasal yang sama & skor kemiripan tertinggi
    # Ini digunakan sebagai acuan ideal pengujian retrieval
    test_query = row['ringkasan_fakta']
    retrieved_results = retrieve(test_query, k=3)
    
    # Kumpulkan id kasus di Train yang pasalnya cocok sebagai ground-truth
    ground_truth_ids = [res["case_id"] for res in retrieved_results if res["pasal_asli"] == row['pasal']]
    
    queries_eval.append({
        "query_id": int(row['case_id']),
        "query_text": test_query,
        "true_pasal": row['pasal'],
        "ground_truth_case_ids": ground_truth_ids if ground_truth_ids else [retrieved_results[0]["case_id"]]
    })

# Simpan ke folder /data/eval/queries.json sesuai panduan tugas
json_output_path = os.path.join(EVAL_FOLDER, "queries.json")
with open(json_output_path, 'w', encoding='utf-8') as f:
    json.dump(queries_eval, f, indent=4, ensure_ascii=False)

print(f"\nFile pengujian berhasil dibuat di: {json_output_path}")

# =====================================================================
# 6. DEMO PENGUJIAN AWAL FUNGSI RETRIEVAL
# =====================================================================
print("\n--- MENCOBA PROSES RETRIEVAL (CONTOH KASUS BARU) ---")
sample_query = "Terdakwa dengan sengaja dan dengan rencana terlebih dahulu merampas nyawa orang lain dengan menusuk korban"
top_cases = retrieve(sample_query, k=3)

for rank, case in enumerate(top_cases):
    print(f"Peringkat {rank+1}: ID Kasus {case['case_id']} | {case['no_perkara']} | {case['pasal_asli']} | Skor Kemiripan: {case['similarity_score']:.4f}")

Jumlah Kasus Lama di Case Base (Train): 32
Jumlah Kueri Kasus Baru untuk Uji Coba (Test): 8

File pengujian berhasil dibuat di: d:\Kuliah\Semester 6\Computer Reasoning\Sub-CPMK3\cbr-pembunuhan\notebooks\data\eval\queries.json

--- MENCOBA PROSES RETRIEVAL (CONTOH KASUS BARU) ---
Peringkat 1: ID Kasus 38 | Tidak Terdeteksi | Pasal 340 KUHP (Pembunuhan Berencana) | Skor Kemiripan: 0.2149
Peringkat 2: ID Kasus 37 | Tidak Terdeteksi | Pasal 340 KUHP (Pembunuhan Berencana) | Skor Kemiripan: 0.2149
Peringkat 3: ID Kasus 27 | Tidak Terdeteksi | Pasal 340 KUHP (Pembunuhan Berencana) | Skor Kemiripan: 0.1911
